In [1]:
import numpy as np  # numpy 패키지 가져오기
import pandas as pd # pandas 패키지 가져오기
import matplotlib.pyplot as plt # 시각화 패키지 가져오기
import seaborn as sns # 시각화

## 2.데이터 전처리
import re                             # 정규식 모듈 임포트

## 3.형태소 처리
from konlpy.tag import Okt
import nltk
from collections import Counter

In [10]:
PBL_df = pd.read_csv("C:\한화에어로스페이스\workspaces\Crawling project\my_projcet\단어분류\단어분류data\Total 물품 낙찰 결과.csv", encoding="cp949")
PBL_df.head()

,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,업무구분,발주기관코드,발주기관,개찰일시,낙찰자(업체코드),낙찰자(상호),낙찰자(영문상호),낙찰자(대표자),낙찰자(연락처),ss,낙찰자(연락처).1,낙찰자(주소),낙찰률,낙찰금액
0,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,물품,SFL,제3283부대,2016-12-30 12:30,H84F8,주식회사 전우밀리터리,Jung woo military,이정구,222730487,222730487,02-2273-0487,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,"23,689,950"
1,1,LFC0043,1,45002,1,장거리 이동 신병 도시락 제조납품,물품,LFC,육군훈련소,2016-12-30 10:00,GA6E7,주식회사 엘엔에프 농업회사법인,LNF,이용환,437318109,437318109,043-731-8109,충청북도 옥천군 옥천읍 삼청리877-1,96.8333,"5,810"
2,22,LGT0034,1,48136,***,0부대 중대시설 분전반 제조설치(16-312),물품,LGT,제2307부대,2016-12-29 11:00,F91A4,설악테크,seorak tech,김화성,334610399,334610399,033-461-0399,강원특별자치도 인제군 북면 원통로74번길10-5(원통농공단지),88.0408,"29,098,720"
3,22,UMM0839,1,48049,***,13-본-국대-01 00학교 가구 및 비품 제조설치,물품,UMM,국군재정관리단,2016-12-29 11:00,B30CC,주식회사 디에스나이키,"DS NAIKI CO.,LTD",김준홍,27342125,27342125,02-734-2125,"서울특별시 구로구 구로중앙로134, 4층(구로동, 리치몰)",88.2297,"2,165,042,000"
4,9,LGT0035,1,48138,***,00부대 중대시설 조명기구 제조납품(16-314),물품,LGT,제2307부대,2016-12-29 11:00,C959B,주식회사 한서,"HANSEO Co.,LTD",이명옥,337462110,337462110,033-746-2110,강원특별자치도 원주시 태장공단길47-0 (태장동),88.3713,"32,424,410"


In [11]:
PBL_df.columns

Index(['참가수', '공고번호', '공고차수', '판단번호', '항목번호', '입찰명', '업무구분', '발주기관코드', '발주기관',
       '개찰일시', '낙찰자(업체코드)', '낙찰자(상호)', '낙찰자(영문상호)', '낙찰자(대표자)', '낙찰자(연락처)',
       'ss', '낙찰자(연락처).1', '낙찰자(주소)', '낙찰률', ' 낙찰금액 '],
      dtype='object')

In [12]:
PBL_T_df = PBL_df.drop(['참가수', '공고번호', '공고차수', '판단번호', '항목번호', '업무구분', '발주기관코드', '발주기관',
       '개찰일시', '낙찰자(업체코드)', '낙찰자(상호)', '낙찰자(영문상호)', '낙찰자(대표자)', '낙찰자(연락처)',
       'ss', '낙찰자(연락처).1', '낙찰자(주소)', '낙찰률', ' 낙찰금액 '], axis=1)
PBL_T_df.head()

,입찰명
0,임시명찰 등 2품목 구매
1,장거리 이동 신병 도시락 제조납품
2,0부대 중대시설 분전반 제조설치(16-312)
3,13-본-국대-01 00학교 가구 및 비품 제조설치
4,00부대 중대시설 조명기구 제조납품(16-314)


In [13]:
len(PBL_T_df)

74732

In [ ]:
# #입찰결과_물품_입찰공고명 .csv
# file_name = '입찰결과_입찰명.csv'
# PBL_T_df.to_csv(file_name, index=False, encoding='cp949')

In [ ]:
#GEmini
import pandas as pd
import re

# 1. CSV 파일 읽기
df = pd.read_csv('1.입찰결과_입찰명.csv', encoding='cp949')

# 2. pick_list와 remove_list를 사용하는 카테고리별 키워드 사전 정의
category_keywords = {
    '1종': {
        'pick_list': ['양곡', '잡곡', '쌀', '보리', '콩', '수육', '돼지고기', '삼겹살', '소고기', '소갈비', '닭고기', '오리고기', '어패류', '생선', '오징어', '새우', '수산물', '소채류', '채소', '야채', '농산물', '김치', '두채류', '두부', '콩나물', '조미료', '장류', '고추장', '된장', '간장', '소금', '설탕', '식용유', '부식', '축산물', '식자재', '가공식품', '냉동식품', '냉장식품', '김', '젓갈', '단무지', '감자', '양파', '마늘', '계란', '달걀', '밀가루', '튀김가루', '분식', '라면', '국수', '우유', '주스', '음료', '과일', '증식', '특식', '건빵', '빵류', '빵', '떡류', '전투식량', '특전식량', '구명식량', '작전식량', '도시락', '반찬', '음료수', '요구르트', '치즈', '햄', '소시지', '생수', '커피', '차류', '다과', '과자류', '카스타드', '초코파이'],
        'remove_list': ['용역', '급식', '식당', '취사', '조리', '용기', '포장', '연료', '가스', '정수', '장비', '물자', '시설', '차량', '함정', '항공기', '의무', '의약품', '피복', '침구', '비품', '건설', '자재', '수리', '부속', '공구', '탄약', '화물', '운송', '보관']
    },
    '2종': {
        'pick_list': ['피복', '전투복', '운동복', '방한복', '침구', '침낭', '모포', '개인장구', '방탄헬멧', '수통', '천막', '텐트', '부대비품', '가구', '책상', '의자', '관물대', '사무용품', '사무기기', '생활용품', '소모품', '명찰', '부대기재', '법무', '군사경찰', '인쇄', '군악', '악기', '군종', '공보정훈', '교육', '체육', '체육기구', '운동기구', '교구', '교재', '함정전용품', '부대기구', '취사기구', '난방기구', '냉방기구', '부대장구', '항공장구', '잠수장구', '화생방', '방독면', '보호의', '제독', '전지', '건전지', '축전지', '야전선', '광케이블', '무전기', '전화기', '전산', '네트워크', '서버', 'PC', '컴퓨터', '노트북', '프린터', '모니터', '소프트웨어', '공병', '소화기', '소화전', '소방호스', '소방용품', '군사지도', '지도', '위성사진', '위장망', '로프', '포장재', '컨테이너', '소형드론', '저가드론', '비품', '작업복', '안전화', '문구', '서류', '깃발', '현수막', '배너', '상패', '청소용품', '세제', '휴지', '수건', '보급품', '잡화'],
        'remove_list': ['용역', '소총', '권총', '기관총', '화기', '무기', '탄약', '유도탄', '폭탄', '자폭드론', '전술드론', '정찰드론', '전차', '장갑차', '항공기', '헬기', '함정', '레이더', '통제장비', '의약품', '의료기기', '시멘트', '철근', '목재', '페인트', '휘발유', '경유', 'LPG', '엔진오일', '수리부속', '정비', '공구', '시스템', '체계', '플랫폼', '고가', '전문', '전투', '임무', '작전', '연료']
    },
    '3종': {
        'pick_list': ['경유', '휘발유', '등유', '제트유', '항공유', 'AV-GAS', '윤활유', '엔진오일', '기어오일', '유압유', '그리스', '방청유', '절삭유', '부동액', '화공약품', '솔벤트', '에탄올', '메탄올', '정수약품', '방역약품', '소독제', '가스', 'LPG', 'LNG', 'CNG', '냉매가스', '산소', '질소', '아세틸렌', '프로판', '부탄', '고체연료', '유류포장재', '드럼', '공드럼', '유류저장탱크', '주유기', '연료', '난방연료', '취사연료', '난방유', '보일러유', '산업용가스'],
        'remove_list': ['용역', '페인트', '도료', '락카', '시너', '의약품', '의료용가스', '식용유', '차량', '항공기', '함정', '장비', '수리', '부속', '건설', '자재', '도로', '포장', '아스팔트', '아스콘', '소화기', '폭약', '추진제', '의료용']
    },
    '4종': {
        'pick_list': ['목재', '합판', '각목', '방부목', '철근', '철골', '강판', '못', '철선', '시멘트', '레미콘', '콘크리트', '골재', '모래', '자갈', '벽돌', '블록', '페인트', '도료', '락카', '바니시', '에나멜', '수성페인트', '유성페인트', '장판', '데코타일', '벽지', '타일', '수도자재', '파이프', '배관', '밸브', '수도꼭지', '전기자재', '전선', '케이블', '차단기', '분전반', '조명기구', 'LED', '철조망', '펜스', '축성공구', '삽', '곡괭이', '건축', '토목', '시설', '보수', '방수', '단열재', '유리', '창호', '문짝', '건축자재', '토목자재', '설비자재', '보일러', '펌프', '환풍기', '전기설비', '통신설비', '소방설비', '합성수지', '아스콘', '보도블럭'],
        'remove_list': ['용역', '엔진오일', '기어오일', '윤활유', '그리스', 'LPG', '가스', '연료', '완성품', '건물', '시설물', '장비', '차량', '전차', '장갑차', '공구', '수리부속', '정비자재', '전자부품', '통신장비', '도로포장', '교량', '항만']
    },
    '5종': {
        'pick_list': ['소구경탄', '실탄', '공포탄', '예광탄', '박격포탄', '포병탄', '고폭탄', '철갑탄', '유도탄', '미사일', '로켓탄', '수류탄', '지뢰', '대인지뢰', '대전차지뢰', '연막탄', '섬광탄', '신호탄', '조명탄', '폭약', 'TNT', 'C4', '신관', '뇌관', '추진제', '함포탄', '수중탄', '기뢰', '폭뢰', '항공탄', '일반폭탄', '확산탄', '자폭드론', '군집자폭드론', '탄두', '탄피', '교보재용탄약', '화공품'],
        'remove_list': ['용역', '소총', '권총', '박격포', '자주포', '방사포', '발사대', '발사관', '화기', '총기', '무기', '무기체계', '레이더', '표적', '사격', '훈련', '정찰드론', '감시드론', '항공기', '헬기', '함정', '수리', '정비', '부속', '공구', '연료', '화공약품', '소화기', '신호등']
    },
    '6종': {
        'pick_list': ['PX', '피엑스', '복지매장', '마트', '매점', '면세품', '면세양주', '면세담배', '과자', '스낵', '컵라면', '아이스크림', '화장품', '세면도구', '생활용품', '스포츠용품', '문구', '완구', '서적', '음반', '의류', '가전제품', '주류'],
        'remove_list': ['용역', '전투식량', '전투복', '군장', '군화', '총기', '탄약', '무기', '장비', '유류', '건설자재', '의약품', '수리부속', '군용', '전술', '작전', '훈련', '정비', '식자재']
    },
    '7종': {
        'pick_list': ['화력', '개인화기', '공용화기', '소총', '권총', '기관총', 'K2', 'K3', '화포', '자주포', '견인포', '박격포', '함포', '함정병기', '수중병기', '사격기재', '함정전투체계', '폭발물처리장비', '특수무기', '방공유도무기', '대공화기', '대전차유도무기', '지대지무기', '방공통제장비', '해상유도무기', '기동', '전차', '장갑차', '일반차량', '특수차량', '차량', '작업차', '고소작업대', '다목적작업차', '트레일러', '항공', '전투임무기', '공중기동기', '헬기', '헬리콥터', '감시통제기', '훈련기', '함정', '전투함', '상륙함', '지원함', '잠수함', '통신전자', '전술통신체계', '암호장비', '레이더', '레이다', '항법장비', '전자전장비', '일반장비', '감시장비', 'CCTV', '드론', '무인항공기', '정찰드론', '교육훈련장비', '시뮬레이터', '정밀측정장비', '시험장비', '측정기', '지원장비', '기동장비', '통신장비', '전자장비', '광학장비', '감시정찰', '무인기', '함선', '무기체계', '통신체계', '전산체계', '플랫폼'],
        'remove_list': ['용역', '소화기(소방)', '분말소화기', '소방호스', '소방', '탄약', '포탄', '유도탄', '미사일', '폭탄', '수리부속', '정비부품', '엔진', '타이어', '공구', '정비', '유지보수', '연료', '항공유', '경유', '윤활유', 'LPG', '건설자재', '시멘트', '철근', '목재', '의약품', '의료기기', '붕대', '소모품', '일반물자', '피복', '침구', '전지', '건전지', '소프트웨어(단독)', '서버(단독)', '구급차', '앰뷸런스']
    },
    '8종': {
        'pick_list': ['의무장비', '외과장비', '치과장비', '방사선장비', '진단장비', 'X-RAY', '엑스레이', 'CT', 'MRI', '초음파진단기', '내시경', '의무물자', '의약품', '백신', '주사기', '항생제', '소염제', '붕대', '거즈', '반창고', '소독약', '위생재료', '의무셋', '들것', '구급차', '앰뷸런스', '혈액', '수액', '의료용가스', '의료소모품', '진단시약', '방역물품', '의료기기', '의료장비', '환자복'],
        'remove_list': ['용역', '정수약품', '방역약품(일반)', '공업용가스', '산소(공업용)', '화공약품', '솔벤트', 'LPG', '피복', '침구', '장비', '차량(일반)', '함정', '항공기', '무기', '탄약', '수리', '부속', '정비', '공구', '식품', '식량']
    },
    '9종': {
        'pick_list': ['수리부속', '정비부품', '예비부품', '부품', '부속품', '교환부품', '엔진', '변속기', '밋션', '차축', '타이어', '배터리', '밧데리', '필터', '에어필터', '오일필터', '베어링', '볼트', '너트', '나사', '가스켓', '브레이크', '패드', '라이닝', '정비자재', '용접봉', '사포', '특수공구', '일반공구', '공구', '공구세트', '드라이버', '스패너', '렌치', '플라이어', '드릴', '절단기', '연마기', '용접기', '계측기', '오실로스코프', '멀티미터', '캘리퍼스', '유도탄수리부속', '정비', '유지보수', '수리키트', '오버홀', '창정비', '기계부품', '전자부품', '전기부품', '윤활장비', '세척장비'],
        'remove_list': ['용역', '완제품', '완성장비', '전차(완성)', '장갑차(완성)', '항공기(완성)', '함정(완성)', '차량(완성)', '레이더(완성)', '무기체계', '총기', '화포', '탄약', '포탄', '유도탄(완제품)', '건설자재', '원자재', '철근', '시멘트', '목재', '페인트', '유류', '연료', '엔진오일(보급용)', '의약품', '피복', '일반물자', '소프트웨어', '시스템개발']
    },
    '10종': {
        'pick_list': ['기타', '미분류', '기타물자', '인쇄물', '도서', '영상', '음향', '홍보물', '기념품', '상용', '일반', '컨설팅', '연구', '조사', '설계', '감리', '폐기물', '처리'],
        'remove_list': ['용역', '쌀', '전투식량', '전투복', '소화기', '휘발유', '경유', 'LPG', '시멘트', '철근', '페인트', '유도탄', '수류탄', '탄약', 'PX', '복지매장', '전차', '장갑차', '레이더', '의약품', '붕대', '수리부속', '정비부품', '엔진', '타이어', '공구', '장비', '시스템', '체계']
    }
}

# 3. pick/remove list 기반으로 카테고리를 분류하는 새로운 함수 정의
def classify_item_advanced(title):
    """
    입찰명을 입력받아 pick_list와 remove_list 규칙에 따라 카테고리를 반환하는 함수.
    """
    if not isinstance(title, str):
        return '분류불가(텍스트아님)'

    # 사전 순서대로(1종부터) 카테고리 규칙을 확인
    for category, rules in category_keywords.items():
        pick_list = rules['pick_list']
        remove_list = rules['remove_list']

        # pick_list의 키워드 중 하나라도 포함하는지 확인
        pick_pattern = '|'.join(pick_list)
        if re.search(pick_pattern, title):
            # 만약 포함한다면, remove_list의 키워드가 하나라도 포함되는지 확인
            if remove_list:
                remove_pattern = '|'.join(remove_list)
                if re.search(remove_pattern, title):
                    continue # remove_list 키워드가 있으면 다음 카테고리로 넘어감
            
            # remove_list에 걸리지 않았다면 현재 카테고리로 분류 확정
            return category
            
    return '미분류' # 모든 규칙에 맞지 않는 경우

# 4. '입찰명'에 새로운 분류 함수를 적용하여 '카테고리' 열 추가
if '입찰명' in df.columns:
    df['카테고리'] = df['입찰명'].apply(classify_item_advanced)
    
    # 5. 결과 확인 및 파일 저장
    print("===== 카테고리 분류 결과 (상위 10개) =====")
    print(df[['입찰명', '카테고리']].head(10))

    print("\n===== 카테고리별 개수 요약 =====")
    print(df['카테고리'].value_counts())
    
    # 분류 결과를 새로운 CSV 파일로 저장
    output_filename = '물품_카테고리_분류완료_제미나이3.csv'
    df.to_csv(output_filename, index=False, encoding='cp949')
    
    print(f"\n분류가 완료되었습니다. 결과가 '{output_filename}' 파일로 저장되었습니다.")
    
else:
    print("오류: '입찰명' 열을 찾을 수 없습니다. CSV 파일의 열 이름을 확인해주세요.")
    print("현재 파일의 열 목록:", df.columns.tolist())

===== 카테고리 분류 결과 (상위 10개) =====
                            입찰명 카테고리
0                 임시명찰 등 2품목 구매   2종
1            장거리 이동 신병 도시락 제조납품   1종
2     0부대 중대시설 분전반 제조설치(16-312)   4종
3  13-본-국대-01 00학교 가구 및 비품 제조설치   2종
4   00부대 중대시설 조명기구 제조납품(16-314)   4종
5                    2017년 난방연료   3종
6    00부대 체육시설 비품구매(고소작업대 등 9종)   2종
7            17년 액화석유가스(LPG) 구매   3종
8                      군수처 취사연료   3종
9            우갈비 등 15품목 (덕산스포텔)  미분류

===== 카테고리별 개수 요약 =====
카테고리
미분류    40999
2종      8889
9종      6475
4종      5328
7종      3807
1종      3604
10종     2067
3종      1729
8종      1295
6종       317
5종       222
Name: count, dtype: int64

분류가 완료되었습니다. 결과가 '물품_카테고리_분류완료_제미나이3.csv' 파일로 저장되었습니다.


In [ ]:
#GPT
import pandas as pd
import re

# 1. CSV 파일 읽기
df = pd.read_csv('1.입찰결과_입찰명.csv', encoding='cp949')

# 2. pick_list와 remove_list를 사용하는 카테고리별 키워드 사전 정의
category_keywords ={
    '1종': {
        'pick_list': ['양곡','잡곡','백미','현미','보리','농산물','축산물','수산물','수육','돈육','우육','계육','어패류','생선','오징어','새우','해조류','김','김치','김치류','장류','고추장','된장','간장','두부','두채류','콩나물','채소','과일','사과','바나나','감자','고구마','양파','배추','무','조미료','소금','설탕','소맥분','밀가루','튀김가루','분식','분식류','라면','국수','면류','컵라면','즉석밥','레토르트식품','도시락','반찬','후식','우유','가공유','요구르트','치즈','주스','음료','음료수','생수','탄산음료','커피믹스','시리얼','빵류','과자류','스낵류','떡류','건빵','전투식량','특전식량','구명식량','식용유','카놀라유','해바라기유','참기름','들기름','소스류','가공식품','냉동식품','냉장식품','냉동만두','식자재'],
        'remove_list': ['경유','휘발유','등유','제트유','항공휘발유','AV-GAS','엔진오일','기어오일','윤활유','그리스','부동액','요소수','LPG','LNG','CNG','산소','질소','아르곤','헬륨','이산화탄소','냉매','R134a','R410a','R32','R22','시멘트','레미콘','콘크리트','철근','철선','철조망','타일','배관자재','밸브','펌프','유도탄','포병탄','소구경탄','자폭용드론','레이더','전술통신체계','지휘통제장비','정밀측정장비','의약품','의약외품','방사선장비','수리부속','용접봉','CCTV통합관제','장비','부품','시스템','체계','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','전산','도서','비품','자재','용품','가스']
    },
    '2종': {
        'pick_list': ['피복','피복류','방한피복','작업복','속옷','군복','군화','전투화','양말','벨트','모포','침구','침낭','개인장구','군장','수통','방한모','우의','방수복','생활용품','취사도구','식기류','위장망','로프','포장재','산업안전용품','안전모','작업용장갑','면장갑','소방피복','소화기','분말소화기','이산화탄소소화기','소화기충전','소화기점검표','오염표지판','야전선','전화기(상용)','상용무전기','휴대용무전기','전지','건전지','배터리','군사지도','지도','육도','해도','항공도','교범','전술교범','용지','복사용지','프린터용지','라벨지','감열지','봉투','문구류','볼펜','지우개','노트','프린터','복합기','스캐너','플로터','문서세단기','세단기','토너','잉크카트리지','리본카트리지','전산소모품','키보드','마우스','허브(소형)','공유기(소형)','AP(소형)','케이블','랜케이블','광케이블','케이블타이','소형드론','민수드론','교육용드론','토이드론','명찰','휘장','계급장','청소기'],
        'remove_list': ['드론','자폭용드론','전술드론','정찰드론','경유','휘발유','등유','제트유','AV-GAS','엔진오일','기어오일','윤활유','그리스','부동액','요소수','LPG','LNG','CNG','냉매','시멘트','레미콘','콘크리트','철근','도료','페인트','타일','분전반','전선(건설)','배관자재','밸브(건설)','펌프(건설)','유도탄','포병탄','전차','장갑차','헬기','항공기','함정','레이더','전술통신체계(군)','서버','스토리지','백본스위치','스위치(네트워크)','라우터','방화벽','UTM','IDS','IPS','지휘통제장비','정밀측정장비','의약품','방사선장비','수리부속','용접봉','CCTV통합관제','관제시스템','무기체계','탄약','체계','장비(고가)','플랫폼','작전','임무','연료','전산장비']
    },
    '3종': {
        'pick_list': ['경유','휘발유','등유','제트유','항공휘발유','AV-GAS','연료유','등유(난방)','엔진오일','기어오일','유압작동오일','압축기오일','터빈오일','윤활유','그리스','방청유','방청제','세척제','세정제','솔벤트','부동액','요소수','AdBlue','제빙제','염화칼슘','정수약품','오폐수처리약품','방역약품','살충제','소독제','멸균제','에탄올','메탄올','이소프로필알코올','IPA','아세톤','MEK','톨루엔','크실렌','염산','황산','수산화나트륨','가성소다','차아염소산나트륨','차아염소산수','LPG','엘피지','LNG','엘엔지','CNG','씨엔지','산소','질소','아르곤','헬륨','이산화탄소','수소','아세틸렌','프로판','부탄','냉매','R134a','R410a','R32','R22','냉매오일','공드럼','유류드럼','IBC탱크','제리캔','가스용기','연료첨가제'],
        'remove_list': ['전투식량','우유','주스','건빵','면류','도시락','시멘트','레미콘','콘크리트','철근','목재','페인트','도료','장판','벽지','철조망','분전반','배관자재','밸브','펌프','유도탄','포병탄','자폭용드론','전차','장갑차','헬기','레이더','전술통신체계','지휘통제장비','정밀측정장비','의약품','의료용가스','PX','수리부속','용접봉','CCTV통합관제','장비','부품','시스템','체계','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','전산','도서','자재','용품']
    },
    '4종': {
        'pick_list': ['시멘트','레미콘','콘크리트','레미탈','몰탈','철근','H형강','형강','철선','철망','못','와이어로프(건설)','목재','합판','석재','벽돌','블록','석고보드','타일','장판','벽지','도배지','페인트','도료','방청도료','프라이머','실란트','실리콘(건설)','방수재','우레탄방수','방수시트','방수테이프','비계자재','거푸집','흙막이','골재','모래','자갈','배관자재','수도자재','동관','PVC배관','전선','전기자재','분전반','차단기','스위치(건설)','콘센트(건설)','분전반함','조명기구(건설)','LED등기구(건설)','철조망','울타리자재','방책자재','축성공구','유량계(건설)','게이트밸브(건설)','볼밸브(건설)','체크밸브(건설)','배수펌프(건설)','창호','유리','문짝'],
        'remove_list': ['경유','휘발유','등유','제트유','AV-GAS','엔진오일','기어오일','윤활유','그리스','부동액','요소수','전투식량','우유','주스','유도탄','포병탄','자폭용드론','전차','장갑차','헬기','레이더','전술통신체계','지휘통제장비','정밀측정장비','의약품','PX','수리부속','용접봉','CCTV통합관제','관제시스템','서버','방화벽','스위치(네트워크)','라우터','장비','부품','시스템','체계','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','전산','도서']
    },
    '5종': {
        'pick_list': ['소구경탄','직사화기탄','기관포탄','박격포탄','포병탄','유도탄','미사일','로켓탄','항공로켓탄','수류탄','지뢰','대인지뢰','대전차지뢰','연막탄','섬광탄','신호탄','조명탄','화학탄','폭약','폭파기재','도폭선','도화선','뇌관','추진장약','점화장약','훈련탄','연습탄','공포탄','함포탄','수중탄','기뢰','폭뢰','RBOC','DAGAIE','신관','자폭용드론','군집자폭드론','탄두','탄피','탄약포장재','탄약상자','탄약통','비축원자재(탄약)','탄약정비재료','교보재용연막제'],
        'remove_list': ['경유','휘발유','등유','제트유','AV-GAS','엔진오일','기어오일','윤활유','그리스','부동액','요소수','시멘트','레미콘','콘크리트','철근','목재','페인트','전투식량','우유','주스','전차','장갑차','헬기','레이더','전술통신체계','지휘통제장비','정밀측정장비','의약품','PX','수리부속','용접봉','CCTV통합관제','야전선','소화기','전산소모품','장비','부품','시스템','체계','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','도서']
    },
    '6종': {
        'pick_list': ['PX','피엑스','복지매장','군마트','장병매점','PX물품','PX납품','PX간식','PX음료','PX과자','PX생활용품','PX세면도구','PX문구','PX의류','PX양말','PX운동화','PX세제','PX샴푸','PX물티슈','PX화장지','PX생수','PX탄산음료','PX커피믹스','PX초콜릿','PX빵','PX라면','PX컵라면','PX건전지','PX칫솔','PX치약','PX기프트카드','PX상품권','PX전용바코드','복지사업단','매장진열대','POS소모품','쇼핑백(PX)','매대진열대'],
        'remove_list': ['전투식량','우유','주스','라면','과자','생수','경유','휘발유','등유','제트유','AV-GAS','엔진오일','시멘트','철근','유도탄','포병탄','자폭용드론','전차','장갑차','헬기','레이더','전술통신체계','지휘통제장비','정밀측정장비','의약품','수리부속','용접봉','CCTV통합관제','관제시스템','장비','부품','시스템','체계','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','도서']
    },
    '7종': {
        'pick_list': ['전차','장갑차','자주포','야포','다련장','전투임무기','공중기동기','헬기','감시통제기','훈련기','함정','전투함','전투지원함','전투근무지원정','잠수장비','항해장비','기관장비','발전기','무정전전원장치','UPS','항온항습기','보일러','공기청정기','정수기','레이더','레이다','소나','음향탐지기','전자전장비','항법장비','GPS수신기(군용)','열상장비','열영상장비','야시경','NVG','레이저거리측정기','관측장비','전술통신체계','전산장비','서버','스토리지','백본스위치','스위치(네트워크)','라우터','방화벽','UTM','IDS','IPS','지휘통제장비','기상장비','정보수집장비','해양장비','암호장비','CCTV통합관제','영상관제시스템','관제서버','NMS','사격기재','폭발물처리장비','EOD장비','도하장비','특전장비','근무지원장비','화생방장비(장비)','병참장비','인쇄장비','출판장비','정밀측정장비','드론체계','UAV체계','지상통제장비','무기체계','특수차량','살수차','청소차','제설차'],
        'remove_list': ['드론','소구경탄','포병탄','유도탄','자폭용드론','경유','휘발유','등유','제트유','AV-GAS','엔진오일','윤활유','그리스','부동액','요소수','시멘트','레미콘','콘크리트','철근','목재','페인트','전투식량','우유','주스','의약품','정수약품','PX','수리부속','용접봉','야전선','상용무전기','소화기','군사지도','복사용지','라벨지','분전반(건설)','전선(건설)','배관자재','밸브(건설)','펌프(건설)','케이블(소형)','청소기','관제시스템(비군)','장비','부품','시스템','체계(일반)','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','도서']
    },
    '8종': {
        'pick_list': ['의무','의무장비','의무물자','의무소모품','의무수리부속','의무특수공구','외과장비','치과장비','방사선장비','진단장비','X선장비','X-RAY','엑스레이','CT장비','MRI장비','초음파진단기','내시경','의약품','의약외품','백신','진단시약','시약','주사기','주사바늘','수액','혈액','소독약','멸균제','멸균기','소독기','의무셋','의무카트','들것','구급','구급함','구급낭','구급키트','AED','제세동기','인공호흡기','전동식수술대','마취기','위생재료','석고붕대','거즈','반창고','의료용밴드','수술가운','수술포','소독포','의료폐기물용기','병원장비','병리장비','환자감시장치'],
        'remove_list': ['소구경탄','포병탄','유도탄','자폭용드론','경유','휘발유','등유','제트유','항공휘발유','AV-GAS','엔진오일','윤활유','그리스','부동액','요소수','시멘트','레미콘','콘크리트','철근','목재','페인트','전투식량','우유','주스','PX','수리부속','용접봉','레이더','전술통신체계','지휘통제장비','정밀측정장비(비의무)','CCTV통합관제','관제시스템','의료용산소(가스)','장비','부품','시스템','체계','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','도서']
    },
    '9종': {
        'pick_list': ['수리부속','정비자재','예비부품','부속품','교환부품','특수공구','일반공구','공구세트','드라이버','스패너','렌치','플라이어','토크렌치','드릴','절단기','연마기','용접기','용접봉','납땜와이어','납땜봉','사포','그라인딩휠','디스크','버프','베어링','패킹','오링','가스켓','씰','패스너','볼트','너트','와셔','리벳','체인','벨트(구동)','브레이크패드','전선단자','단자대','케이블타이','전구','램프','LED전구','퓨즈','릴레이','전구(정비)','램프(정비)','LED모듈(정비)','계측기','오실로스코프','멀티미터','캘리퍼스','필터','필터카트리지','에어필터','오일필터','연료필터','점화플러그','호스(정비)','클램프','수리킷','유도탄수리부속','항공수리부속','통신전자수리부속','정밀측정장비수리부속','연료펌프(부속)','워터펌프(부속)','밸브코어(부속)','함정용특수로프','함정전용전기자재','세면장자재'],
        'remove_list': ['경유','휘발유','등유','제트유','항공휘발유','AV-GAS','엔진오일','윤활유','그리스','부동액','요소수','시멘트','레미콘','콘크리트','철근','목재','페인트(건설)','전투식량','우유','주스','소구경탄','포병탄','유도탄','자폭용드론','전차','장갑차','헬기','항공기','함정(완성)','레이더(완성)','전술통신체계','지휘통제장비','의약품','PX','CCTV통합관제','전산장비','드론체계','분전반(건설)','전선(건설)','배관자재(건설)','밸브(건설)','펌프(건설)','장비(완성)','시스템','체계','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','도서']
    },
    '10종': {
        'pick_list': ['기타','미분류','기타물자'],
        'remove_list': ['양곡','전투식량','도시락','우유','주스','피복','침구','야전선','상용무전기','군사지도','소화기','경유','휘발유','등유','제트유','항공휘발유','엔진오일','요소수','LPG','LNG','CNG','냉매','시멘트','레미콘','콘크리트','철근','철조망','분전반','소구경탄','유도탄','자폭용드론','PX','전차','장갑차','레이더','전술통신체계','지휘통제장비','의약품','수리부속','용접봉','CCTV통합관제','관제시스템','장비','부품','시스템','체계','용역','구매','제작','납품','공급','설치','조달','물품','세트','시험','도서']
    }
}


# 3. pick/remove list 기반으로 카테고리를 분류하는 새로운 함수 정의
def classify_item_advanced(title):
    """
    입찰명을 입력받아 pick_list와 remove_list 규칙에 따라 카테고리를 반환하는 함수.
    """
    if not isinstance(title, str):
        return '분류불가(텍스트아님)'

    # 사전 순서대로(1종부터) 카테고리 규칙을 확인
    for category, rules in category_keywords.items():
        pick_list = rules['pick_list']
        remove_list = rules['remove_list']

        # pick_list의 키워드 중 하나라도 포함하는지 확인
        pick_pattern = '|'.join(pick_list)
        if re.search(pick_pattern, title):
            # 만약 포함한다면, remove_list의 키워드가 하나라도 포함되는지 확인
            if remove_list:
                remove_pattern = '|'.join(remove_list)
                if re.search(remove_pattern, title):
                    continue # remove_list 키워드가 있으면 다음 카테고리로 넘어감
            
            # remove_list에 걸리지 않았다면 현재 카테고리로 분류 확정
            return category
            
    return '미분류' # 모든 규칙에 맞지 않는 경우

# 4. '입찰명'에 새로운 분류 함수를 적용하여 '카테고리' 열 추가
if '입찰명' in df.columns:
    df['카테고리'] = df['입찰명'].apply(classify_item_advanced)
    
    # 5. 결과 확인 및 파일 저장
    print("===== 카테고리 분류 결과 (상위 10개) =====")
    print(df[['입찰명', '카테고리']].head(10))

    print("\n===== 카테고리별 개수 요약 =====")
    print(df['카테고리'].value_counts())
    
    # 분류 결과를 새로운 CSV 파일로 저장
    output_filename = '물품_카테고리_분류완료_GPT2.csv'
    df.to_csv(output_filename, index=False, encoding='cp949')
    
    print(f"\n분류가 완료되었습니다. 결과가 '{output_filename}' 파일로 저장되었습니다.")
    
else:
    print("오류: '입찰명' 열을 찾을 수 없습니다. CSV 파일의 열 이름을 확인해주세요.")
    print("현재 파일의 열 목록:", df.columns.tolist())

===== 카테고리 분류 결과 (상위 10개) =====
                            입찰명 카테고리
0                 임시명찰 등 2품목 구매   2종
1            장거리 이동 신병 도시락 제조납품  미분류
2     0부대 중대시설 분전반 제조설치(16-312)  미분류
3  13-본-국대-01 00학교 가구 및 비품 제조설치  미분류
4   00부대 중대시설 조명기구 제조납품(16-314)  미분류
5                    2017년 난방연료  미분류
6    00부대 체육시설 비품구매(고소작업대 등 9종)  미분류
7            17년 액화석유가스(LPG) 구매  미분류
8                      군수처 취사연료  미분류
9            우갈비 등 15품목 (덕산스포텔)  미분류

===== 카테고리별 개수 요약 =====
카테고리
미분류    64113
2종      5863
1종      1935
9종       997
7종       745
4종       527
3종       235
8종       193
5종       111
10종       13
Name: count, dtype: int64

분류가 완료되었습니다. 결과가 '물품_카테고리_분류완료_GPT2.csv' 파일로 저장되었습니다.


In [25]:
#human
import pandas as pd
import re

# 1. CSV 파일 읽기
df = pd.read_csv('1.입찰결과_입찰명.csv', encoding='cp949')

# 2. pick_list와 remove_list를 사용하는 카테고리별 키워드 사전 정의
category_keywords = {
        '1종':{
             'pick_list':  [
             '빙수', '바나나', '우유', '콜라', '과일', '탄산', '음료', '발효', '젤리', '견과', '케이크', '브레드', 
             '짜장', '밥', '허니', '감자', '스낵', '식용', '핫바', '소시지', '어묵', '핫도그', '두부', '고기', '떡', 
             '쌈무', '메추리', '순대', '단무지', '우동', '단호박', '샐러드', '고구마', '국수', '치즈', '라이스', 
             '스파게티', '사과', '푸딩', '망고', '요거트', '콘푸', '마카로니', '돈까스', '소스', '간장', '초코', 
             '마가렛', '베리', '에너지', '몽쉘', '사이다', '매실', '포도', '식혜', '옥수수', '두유', '요구르트', 
             '복숭아', '아이스티', '코코넛', '농축액', '피자', '머핀', '견과류', '부식', '분식', '채류', '김치', 
             '쿠키', '미니', '약과', '음식', '쇠고기', '콩나물', '아이스크림', '도시락', '과자', '샌드', '양갱', 
             '생수', '후식', '만쥬', '슈크림', '파인애플', '과채', '비타민', '커피', '삼겹', '건포도', '삼계탕', 
             '조림', '깻잎', '말랭이', '땅콩', '춧잎', '장아찌', '절임', '양념', '시래기', '김자', '미역', '튀김', 
             '명란', '젓갈', '창란', '꼴뚜기', '낙지', '깍두기', '키위', '메론', '돼지', '등뼈', '갈비', '해물', 
             '숯불', '닭', '생선', '까스', '너비아니', '만두', '동그랑땡', '꼬치', '스테이크', '치킨', '너겟', '새우', 
             '말이', '탕수육', '날치', '케익크', '고기만', '바베큐', '폭립', '청국장', '맛살', '오리', '훈제', '족발', '칵테일', 
             '통조림', '육수', '버터', '설탕', '차', '가루', '탕', '액젓', '짬뽕', '육류', '동태', '커틀렛', '토란', 
             '오징어', '드레싱', '식초', '찜', '딸기', '식량', '식단', '주스', '토마토', '케첩', '시리얼', '쌀', '감자전', 
             '버섯', '오이', '피클', '가자미', '계란', '농산', '수산', '축산', '고추', '야채', '햄', '후추', '한식', 
             '쌈장', '깨', '조미', '기름', '샌드위치', '볶음', '팝콘', '묵', '육개장', '빵', '미트', '유제품', '황도', 
             '파스타', '주류', '채소', '맛술', '연어', '버팔로', '치킨윙', '해쉬', '포테이토', '스팸', '맥주', '라멘', 
             '구이', '도너츠', '영양식', '푸실리', '생강', '홍합', '꽃게', '바지락', '정육', '원유', '비빔밥', '나물', 
             '카라멜', '죽', '레토르트', '발사믹', '춘장', '다시다', '고로케', '곤약', '베이컨', '체다', '어패류', '깐풍기', 
             '가쓰오부시', '교자', '까르보나라', '찌개', '찹쌀', '육전', '할라피뇨', '소세지', '아몬드', '차돌', '과즙', 
             '고등어', '곡류', '돈육', '오이지', '크로크무슈', '간편식', '애플', '코코아', '스무디', '라떼', '타코야끼', 
             '생크림', '우엉', '배맛', '샘표', '대패', '버팔로윙', '부챗살', '옹심이', '갈릭', '크로와상', '샘물', 
             '새우깡', '비요뜨', '마들렌', '스프', '급식', '환자식', '특식', '식품', '식자재', '증식', '카페', '가다랑어포', 
             '된장', '천일염', '마요네즈', '엿', '시럽', '감귤', '리치', '중식', '전복', '간식', '유자', '패스츄리', '영양', 
             '비스킷', '떡국', '해장국', '재첩국', '쫄면', '자장면', '생면', '냉면', '양곡', '잡곡', '백미', '현미', '보리', 
             '농산물', '축산물', '수산물', '수육', '돈육', '우육', '계육', '해조류', '김', '김치류', '장류', '고추장', 
             '된장', '두채류', '배추', '무', '소금', '소맥분', '밀가루', '튀김가루', '분식류', '라면', '면류', '라면', 
             '즉석밥', '레토르트식품', '반찬', '가공유', '음료수', '탄산음료', '커피믹스', '빵류', '과자류', '스낵류', 
             '떡류', '건빵', '전투식량', '특전식량', '구명식량', '식용유', '카놀라유', '해바라기유', '참기름', '들기름', 
             '소스류', '가공식품', '냉동식품', '냉장식품', '냉동만두', '콩', '돼지고기', '삼겹살', '소고기', 
             '소갈비', '닭고기', '오리고기', '소채류', '마늘', '달걀', '작전식량', '차류', '다과', '카스타드', '초코파이'],
        'remove_list':['용역','기구', '냉동고', '에너지', '장치','제조기', '장비', '압축기', '오리콘', '폐기물', 
                         '작업', '공사', '판넬', '설치', '물자', '장갑',  '솥', '전기', '절단기',
                        '스프링', '주사',  '인버터', '용품', '감량기', '수산화', '가죽', '기계', '버섯형', '기계',
                        '카드', '검사', '방지제', '스위치', '비누', '폭발', '칼', '분쇄기', '락카', '멜빵']
        },
    '2종': {
        'pick_list': [
                "군복","피복","방한","침구","장구","의류","방탄복","군화","모자","양말","티셔츠","속옷","안전화","작업복","전투복","일용품","특수임무피복","비품","기재","취사","냉난방","군악","체육","교육","장구","포장재","실험기구","안전용품","로프","비품류","장비류","기구류","군복","피복","방한","침구","장구","의류","방탄복","군화","모자","양말","티셔츠","속옷","안전화","작업복","전투복","일용품","특수임무피복","베어링","볼트","너트","와셔","패킹","씰","가스켓","체인","기어","필터","패드","브러시","벨트","휠","스프링","법무","군사경찰","인쇄","군종","공보정훈","항공장구","잠수장구","기류","해상표적","탐지","보호장비","방독면","치료물자","연막","제독제","구조기구","오염표지판","지역장비","독성제거제","작도기재","전지","배터리","야전선","CCTV","무전기","전화","팩스","라디오","위성TV","전산","네트워크","컴퓨터","휴대용무전기","수신기","소방","소화기","위장망","군사지도","지도","드론","육도","해도","항공도", '전투사', '잉크', '코인', '베트남', '송풍기', '간행물', '수저'],
        'remove_list': ['용역', '소총', '권총', '기관총', '화기', '무기', '탄약', '유도탄', '폭탄', '자폭드론', '전술드론', '정찰드론', '전차', '장갑차', '항공기', '헬기', '함정', '레이더', '통제장비', '의약품', '의료기기', '시멘트', '철근', '목재', '페인트', '휘발유', '경유', 'LPG', '엔진오일', '수리부속', '정비', '공구', '시스템', '체계', '플랫폼', '고가', '전문', '임무', '작전', '연료', '온수', '어린이집', '탄약', '절연']
        },
    '3종': {
        'pick_list': ["경유","휘발유","등유","제트유","항공유","항공휘발유","AV-GAS","엔진오일","기어오일","유압작동오일","압축기오일","그리스","방청유","방청제","세관제","솔벤트","부동액","제초제","정수약품","방역약품","살충제","목재펠릿","연탄","공탄","조개탄","착화탄","고형연료","LPG","LNG","CNG","아세틸렌","냉매","가스","드럼","드럼통","가스용기", '연료', ],
        'remove_list': ['용역', '페인트', '도료', '락카', '시너', '의약품', '의료용가스', '식용유', '차량', '항공기', '함정', '장비', '수리', '부속', '건설', '자재', '도로', '포장', '아스팔트', '아스콘', '소화기', '폭약', '추진제', '의료용']
    },
    '4종': {
            'pick_list': ['LED', 'PVC배관', '각목', '강관', '강판', '건축', '거푸집', '건설', \
                          '골재', '공사', '곡괭이', '데코타일', '도료', '도배지', '동관', '락카', '레미콘', '레미탈', '모래', '목재', '못', \
                          '몰탈', '문짝', '바니시', '방부목', '방수', '방청도료', '방책자재', '배관', \
                          '배수', '펌프', '벽돌', '벽지', '보도블럭', '보수', '보일러', '볼밸브', '블록', '비계자재', '삽', \
                          '생활관', '석고보드', '석재', '설비자재', '소방설비', '수도꼭지', '수도자재', '스위치', '시멘트', \
                          '시설', '신축', '아스콘', '아스팔트', '아파트', '에나멜', '울타리', '울타리자재', '우레탄방수', '유량계', '유리', \
                          '유성페인트', '자갈', '장판', '제조', '제조 설치', '제조설치', '조명기구', '체크밸브', '축성공구', \
                          '철골', '철망', '철선', '철재', '철조망', '창호', '철근', '콘센트', '콘크리트', '케이블', '타일', '토목', \
                          '통신설비', '판넬', '파이프', '페인트', '펜스', '프라이머', '합성수지', '합판', '형강', '환풍기', '흙막이', 'H형강', \
                          '와이어로프', '실란트', '실리콘', '단열재', '분전반', '분전반함', '차단기', '전기자재', '전기설비', '전선', '밸브', '설치 자재', '설치자재'],
            'remove_list': ['용역', '병원', '의무', '치료', '드론']
    },
    '5종': {
        'pick_list': ["소구경탄","직사화기탄","박격포탄","포병탄","지대지유도탄","수류탄","지뢰","연막탄","신호탄","화학탄","폭약","폭파기재","함포탄","수중탄","기뢰","폭뢰","함정유도탄","RBOC","DAGAIE","항공유도탄","투하탄","일반폭탄","확산탄","기관포탄","항공로켓탄","심리전탄","전자전탄","자폭드론","탄피","탄약포장","비축원자재"],
        'remove_list': ['용역', '소총', '권총', '박격포', '자주포', '방사포', '발사대', '발사관', '화기', '총기', '무기', '무기체계', '레이더', '표적', '사격', '훈련', '정찰드론', '감시드론', '항공기', '헬기', '함정', '수리', '정비', '부속', '공구', '연료', '화공약품', '소화기', '신호등']
    },
    '6종': {
        'pick_list': ['PX', '피엑스', '복지매장', '마트', '매점', '면세품', '면세양주', '면세담배', '과자', '스낵', '컵라면', '아이스크림', '화장품', '세면도구', '생활용품', '스포츠용품', '문구', '완구', '서적', '음반', '의류', '가전제품', '주류'],
        'remove_list': ['용역', '전투식량', '전투복', '군장', '군화', '총기', '탄약', '무기', '장비', '유류', '건설자재', '의약품', '수리부속', '군용', '전술', '작전', '훈련', '정비', '식자재']
    },
    '7종': {
        'pick_list': ['화력', '개인화기', '공용화기', '소총', '권총', '기관총', 'K2', 'K3', '화포', '자주포', '견인포', '박격포', '함포', '함정병기', '수중병기', '사격기재', '함정전투체계', '폭발물처리장비', '특수무기', '방공유도무기', '대공화기', '대전차유도무기', '지대지무기', '방공통제장비', '해상유도무기', '기동', '전차', '장갑차', '일반차량', '특수차량', '차량', '작업차', '고소작업대', '다목적작업차', '트레일러', '항공', '전투임무기', '공중기동기', '헬기', '헬리콥터', '감시통제기', '훈련기', '함정', '전투함', '상륙함', '지원함', '잠수함', '통신전자', '전술통신체계', '암호장비', '레이더', '레이다', '항법장비', '전자전장비', '일반장비', '감시장비', 'CCTV', '드론', '무인항공기', '정찰드론', '교육훈련장비', '시뮬레이터', '정밀측정장비', '시험장비', '측정기', '지원장비', '기동장비', '통신장비', '전자장비', '광학장비', '감시정찰', '무인기', '함선', '무기체계', '통신체계', '전산체계', '플랫폼'],
        'remove_list': ['용역', '소화기(소방)', '분말소화기', '소방호스', '소방', '탄약', '포탄', '유도탄', '미사일', '폭탄', '수리부속', '정비부품', '엔진', '타이어', '공구', '정비', '유지보수', '연료', '항공유', '경유', '윤활유', 'LPG', '건설자재', '시멘트', '철근', '목재', '의약품', '의료기기', '붕대', '소모품', '일반물자', '피복', '침구', '전지', '건전지', '소프트웨어(단독)', '서버(단독)', '구급차', '앰뷸런스']
    },
    '8종': {
        'pick_list': ['병원', '환자', '의무', '외상', '안과', '자극', '신경', '의학', \
             '이비인후과', '치과', '붕대', '외과', '수술', '의료', '병리', '예방약', \
             '방사선장비', '의약품', '치료', '안경제작', '위생'],
        'remove_list': ['신축', '보일러', '제조설치']
    },
    '9종': {
        'pick_list': ['수리부속', '정비부품', '예비부품', '부품', '부속품', '교환부품', '엔진', '변속기', '밋션', '차축', '타이어', '배터리', '밧데리', '필터', '에어필터', '오일필터', '베어링', '볼트', '너트', '나사', '가스켓', '브레이크', '패드', '라이닝', '정비자재', '용접봉', '사포', '특수공구', '일반공구', '공구', '공구세트', '드라이버', '스패너', '렌치', '플라이어', '드릴', '절단기', '연마기', '용접기', '계측기', '오실로스코프', '멀티미터', '캘리퍼스', '유도탄수리부속', '정비', '유지보수', '수리키트', '오버홀', '창정비', '기계부품', '전자부품', '전기부품', '윤활장비', '세척장비'],
        'remove_list': ['용역', '완제품', '완성장비', '전차(완성)', '장갑차(완성)', '항공기(완성)', '함정(완성)', '차량(완성)', '레이더(완성)', '무기체계', '총기', '화포', '탄약', '포탄', '유도탄(완제품)', '건설자재', '원자재', '철근', '시멘트', '목재', '페인트', '유류', '연료', '엔진오일(보급용)', '의약품', '피복', '일반물자', '소프트웨어', '시스템개발']
    },
    '10종': {
        'pick_list': ['기타', '미분류', '기타물자', '인쇄물', '도서', '영상', '음향', '홍보물', '기념품', '상용', '일반', '컨설팅', '연구', '조사', '설계', '감리', '폐기물', '처리', '어린이집'],
        'remove_list': ['용역', '쌀', '전투식량', '전투복', '소화기', '휘발유', '경유', 'LPG', '시멘트', '철근', '페인트', '유도탄', '수류탄', '탄약', 'PX', '복지매장', '전차', '장갑차', '레이더', '의약품', '붕대', '수리부속', '정비부품', '엔진', '타이어', '공구', '장비', '시스템', '체계']
    },
    '11종':{
        'pick_list': ['용역'],
        'remove_list': ['fuck']
    }
}

# 3. pick/remove list 기반으로 카테고리를 분류하는 새로운 함수 정의
def classify_item_advanced(title):
    """
    입찰명을 입력받아 pick_list와 remove_list 규칙에 따라 카테고리를 반환하는 함수.
    """
    if not isinstance(title, str):
        return '분류불가(텍스트아님)'

    # 사전 순서대로(1종부터) 카테고리 규칙을 확인
    for category, rules in category_keywords.items():
        pick_list = rules['pick_list']
        remove_list = rules['remove_list']

        # pick_list의 키워드 중 하나라도 포함하는지 확인
        pick_pattern = '|'.join(pick_list)
        if re.search(pick_pattern, title):
            # 만약 포함한다면, remove_list의 키워드가 하나라도 포함되는지 확인
            if remove_list:
                remove_pattern = '|'.join(remove_list)
                if re.search(remove_pattern, title):
                    continue # remove_list 키워드가 있으면 다음 카테고리로 넘어감
            
            # remove_list에 걸리지 않았다면 현재 카테고리로 분류 확정
            return category
            
    return '미분류' # 모든 규칙에 맞지 않는 경우

# 4. '입찰명'에 새로운 분류 함수를 적용하여 '카테고리' 열 추가
if '입찰명' in df.columns:
    df['카테고리'] = df['입찰명'].apply(classify_item_advanced)
    
    # 5. 결과 확인 및 파일 저장
    print("===== 카테고리 분류 결과 (상위 10개) =====")
    print(df[['입찰명', '카테고리']].head(10))

    print("\n===== 카테고리별 개수 요약 =====")
    print(df['카테고리'].value_counts())
    
    # 분류 결과를 새로운 CSV 파일로 저장
    output_filename = '물품_카테고리_분류완료_인간승리.csv'
    df.to_csv(output_filename, index=False, encoding='cp949')
    
    print(f"\n분류가 완료되었습니다. 결과가 '{output_filename}' 파일로 저장되었습니다.")
    
else:
    print("오류: '입찰명' 열을 찾을 수 없습니다. CSV 파일의 열 이름을 확인해주세요.")
    print("현재 파일의 열 목록:", df.columns.tolist())

===== 카테고리 분류 결과 (상위 10개) =====
                            입찰명 카테고리
0                 임시명찰 등 2품목 구매  미분류
1            장거리 이동 신병 도시락 제조납품   1종
2     0부대 중대시설 분전반 제조설치(16-312)   4종
3  13-본-국대-01 00학교 가구 및 비품 제조설치   2종
4   00부대 중대시설 조명기구 제조납품(16-314)   4종
5                    2017년 난방연료   3종
6    00부대 체육시설 비품구매(고소작업대 등 9종)   2종
7            17년 액화석유가스(LPG) 구매   3종
8                      군수처 취사연료   3종
9            우갈비 등 15품목 (덕산스포텔)   1종

===== 카테고리별 개수 요약 =====
카테고리
미분류    29025
1종     12673
4종     10435
2종     10134
9종      3956
8종      2824
7종      2111
10종     1738
3종      1328
11종      347
5종        92
6종        69
Name: count, dtype: int64

분류가 완료되었습니다. 결과가 '물품_카테고리_분류완료_인간승리.csv' 파일로 저장되었습니다.


In [26]:
df_yetCate = df[df['카테고리']=='미분류']
output_filename = '물품_카테고리_미분류.csv'
df_yetCate.to_csv(output_filename, index=False, encoding='cp949')